In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from time import time
from sklearn import metrics
from sklearn.pipeline import make_pipeline
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.decomposition import PCA

# 1)


In [ ]:
DataSet = 'Data_Cortex_Nuclear.xls'
DataFrame = pd.read_excel(DataSet)
DataFrame = DataFrame.drop('BCL2_N', axis=1)
DataFrame = DataFrame.dropna()
print(DataFrame)

       MouseID  DYRK1A_N   ITSN1_N    BDNF_N     NR1_N    NR2A_N    pAKT_N  \
0        309_1  0.503644  0.747193  0.430175  2.816329  5.990152  0.218830   
1        309_2  0.514617  0.689064  0.411770  2.789514  5.685038  0.211636   
2        309_3  0.509183  0.730247  0.418309  2.687201  5.622059  0.209011   
3        309_4  0.442107  0.617076  0.358626  2.466947  4.979503  0.222886   
4        309_5  0.434940  0.617430  0.358802  2.365785  4.718679  0.213106   
...        ...       ...       ...       ...       ...       ...       ...   
1075  J3295_11  0.254860  0.463591  0.254860  2.092082  2.600035  0.211736   
1076  J3295_12  0.272198  0.474163  0.251638  2.161390  2.801492  0.251274   
1077  J3295_13  0.228700  0.395179  0.234118  1.733184  2.220852  0.220665   
1078  J3295_14  0.221242  0.412894  0.243974  1.876347  2.384088  0.208897   
1079  J3295_15  0.302626  0.461059  0.256564  2.092790  2.594348  0.251001   

       pBRAF_N  pCAMKII_N   pCREB_N  ...   pCFOS_N     SYP_N  H

# 2)

In [ ]:
def bench_k_means(kmeans, name, data, labels):
    """Benchmark to evaluate the KMeans initialization methods.

    Parameters
    ----------
    kmeans : KMeans instance
        A :class:`~sklearn.cluster.KMeans` instance with the initialization
        already set.
    name : str
        Name given to the strategy. It will be used to show the results in a
        table.
    data : ndarray of shape (n_samples, n_features)
        The data to cluster.
    labels : ndarray of shape (n_samples,)
        The labels used to compute the clustering metrics which requires some
        supervision.
    """
    t0 = time()
    estimator = make_pipeline(kmeans).fit(data)
    fit_time = time() - t0
    results = [name, fit_time, estimator[-1].inertia_]

    # Define the metrics which require only the true labels and estimator
    # labels
    clustering_metrics = [
        metrics.homogeneity_score,
        metrics.completeness_score,
        metrics.v_measure_score,
        metrics.adjusted_rand_score,
        metrics.adjusted_mutual_info_score,
    ]
    results += [m(labels, estimator[-1].labels_) for m in clustering_metrics]

    # The silhouette score requires the full dataset
    results += [
        metrics.silhouette_score(
            data,
            estimator[-1].labels_,
            metric="euclidean",
            sample_size=300,
        )
    ]

    results.append(kmeans.n_clusters)
    header = ["init", "time", "inertia", "homo", "compl", "v-meas", "ARI", "AMI", "silhouette", "n_clusters"]
    formatter_header = (
        "{:8s}\t{:4s}\t{:7s}\t{:4s}\t{:5s}\t{:6s}\t{:3s}\t{:3s}\t{:10s}\t{:4s}"
    )
    print(formatter_header.format(*header))
    # Show the results
    formatter_result = (
        "{:8s}\t{:.3f}s\t{:.0f}\t{:.3f}\t{:.3f}\t{:.3f}\t{:.3f}\t{:.3f}\t{:.3f}\t{:9.0f}"
    )
    print(formatter_result.format(*results))

# 3)

In [ ]:
def bench_clustering2(method, name, data, labels, calculate_all):
    """Benchmark to evaluate the DBSCAN and Aglomerative initialization methods.

    Parameters
    ----------
    methods : Clustering instance
        A :class: `~sklearn.cluster.DBSCAN` or `sklearn.cluster.AgglomerativeClustering` instance with the initialization
        already set.
    name : str
        Name given to the strategy. It will be used to show the results in a
        table.
    data : ndarray of shape (n_samples, n_features)
        The data to cluster.
    labels : ndarray of shape (n_samples,)
        The labels used to compute the clustering metrics which requires some
        supervision.
    """
    t0 = time()
    estimator = make_pipeline(method).fit(data)
    fit_time = time() - t0
    results = [name, fit_time]

    # Define the metrics which require only the true labels and estimator
    # labels
    clustering_metrics = [
        metrics.homogeneity_score,
        metrics.completeness_score,
        metrics.v_measure_score,
        metrics.adjusted_rand_score,
        metrics.adjusted_mutual_info_score,
    ]

    if isinstance(method, sklearn.cluster.DBSCAN):
      n_clusters = len(set(estimator[-1].labels_)) - (1 if -1 in estimator[-1].labels_ else 0)
      if calculate_all and -1 in estimator[-1].labels_:
        predicted_labels = np.where(estimator[-1].labels_ == -1, n_clusters, estimator[-1].labels_)
        n_clusters += 1
    elif isinstance(method, sklearn.cluster.AgglomerativeClustering):
      n_clusters = method.n_clusters

    if n_clusters > 1:
      if not calculate_all and isinstance(method, sklearn.cluster.DBSCAN):
        mask = estimator[-1].labels_ != -1
      else:
        mask = np.ones(len(estimator[-1].labels_), dtype=bool)

      results += [m(labels[mask], estimator[-1].labels_[mask]) for m in clustering_metrics]

      # The silhouette score requires the full dataset
      results += [
          metrics.silhouette_score(
              data[mask],
              estimator[-1].labels_[mask],
              metric="euclidean",
              sample_size=300,
          )
      ]

    else:
      print("One or no clusters found")
      results += [np.nan] * 6

    results.append(n_clusters)
    header = ["init", "time", "homo", "compl", "v-meas", "ARI", "AMI", "silhouette", "n_clusters"]
    formatter_header = (
        "{:8s}\t{:4s}\t{:4s}\t{:5s}\t{:6s}\t{:3s}\t{:3s}\t{:10s}\t{:4s}"
    )
    print(formatter_header.format(*header))
    # Show the results
    formatter_result = (
        "{:8s}\t{:.3f}s\t{:.0f}\t{:.3f}\t{:.3f}\t{:.3f}\t{:.3f}\t{:.3f}\t{:10.0f}"
    )
    print(formatter_result.format(*results))

# 4)



In [ ]:
data = DataFrame.drop('class', axis=1).select_dtypes(include=['float64', 'int64']).values
labels = DataFrame['class'].values
KMeans_1 = KMeans(init='random', n_clusters=8, n_init=10)
Kmeans_2 = KMeans(init='k-means++', n_clusters=8 , n_init=10)
DBSCAN_1 = DBSCAN()
DBSCAN_2 = DBSCAN(eps=0.7)
DBSCAN_3 = DBSCAN(eps=0.2)
Agglomerative_1 = AgglomerativeClustering(n_clusters=8)
reduced_data = PCA(n_components=2).fit_transform(data)
KMeans_1_PCA = KMeans_1.fit(reduced_data)
Kmeans_2_PCA = Kmeans_2.fit(reduced_data)
DBSCAN_1_PCA = DBSCAN_1.fit(reduced_data)
DBSCAN_2_PCA = DBSCAN_2.fit(reduced_data)
DBSCAN_3_PCA = DBSCAN_3.fit(reduced_data)
Agglomerative_1_PCA = Agglomerative_1.fit(reduced_data)

## a)

In [ ]:
bench_k_means(KMeans_1, "KMeans_1", data, labels)

init    	time	inertia	homo	compl	v-meas	ARI	AMI	silhouette	n_clusters
KMeans_1	0.051s	1123	0.290	0.291	0.290	0.144	0.276	0.199	        8


## b)


In [ ]:
bench_k_means(Kmeans_2, "Kmeans_2", data, labels)

init    	time	inertia	homo	compl	v-meas	ARI	AMI	silhouette	n_clusters
Kmeans_2	0.557s	1123	0.300	0.300	0.300	0.147	0.286	0.215	        8


## c)

In [ ]:
bench_clustering2(DBSCAN_1, "DBSCAN_1", data, labels, False)

init    	time	homo	compl	v-meas	ARI	AMI	silhouette	n_clusters
DBSCAN_1	0.136s	1	0.705	0.827	0.539	0.771	0.693	        12


## d)


In [ ]:
bench_clustering2(DBSCAN_2, "DBSCAN_2", data, labels, False)

init    	time	homo	compl	v-meas	ARI	AMI	silhouette	n_clusters
DBSCAN_2	0.011s	1	0.536	0.692	0.263	0.630	0.377	        43


## e)

In [ ]:
bench_clustering2(DBSCAN_3, "DBSCAN_3", data, labels, False)

One or no clusters found
init    	time	homo	compl	v-meas	ARI	AMI	silhouette	n_clusters
DBSCAN_3	0.014s	nan	nan	nan	nan	nan	nan	         0


## f)

In [ ]:
bench_clustering2(DBSCAN_1, "DBSCAN_4", data, labels, True)

init    	time	homo	compl	v-meas	ARI	AMI	silhouette	n_clusters
DBSCAN_4	0.016s	0	0.365	0.165	0.002	0.113	-0.332	        13


## g)

In [ ]:
bench_clustering2(DBSCAN_2, "DBSCAN_5", data, labels, True)

init    	time	homo	compl	v-meas	ARI	AMI	silhouette	n_clusters
DBSCAN_5	0.009s	1	0.425	0.490	0.072	0.422	-0.009	        44


## h)

In [ ]:
bench_clustering2(DBSCAN_3, "DBSCAN_6", data, labels, True)

One or no clusters found
init    	time	homo	compl	v-meas	ARI	AMI	silhouette	n_clusters
DBSCAN_6	0.012s	nan	nan	nan	nan	nan	nan	         1


## i)

In [ ]:
bench_clustering2(Agglomerative_1, "Agglomerative_1", data, labels, False)

init    	time	homo	compl	v-meas	ARI	AMI	silhouette	n_clusters
Agglomerative_1	0.020s	0	0.356	0.354	0.168	0.341	0.140	         8


## j)

In [ ]:
bench_k_means(KMeans_1_PCA, "KMeans_1_PCA", reduced_data, labels)
bench_k_means(Kmeans_2_PCA, "Kmeans_2_PCA", reduced_data, labels)
bench_clustering2(DBSCAN_1_PCA, "DBSCAN_1_PCA", reduced_data, labels, False)
bench_clustering2(DBSCAN_2_PCA, "DBSCAN_2_PCA", reduced_data, labels, False)
bench_clustering2(DBSCAN_3_PCA, "DBSCAN_3_PCA", reduced_data, labels, False)
bench_clustering2(DBSCAN_1_PCA, "DBSCAN_4_PCA", reduced_data, labels, True)
bench_clustering2(DBSCAN_2_PCA, "DBSCAN_5_PCA", reduced_data, labels, True)
bench_clustering2(DBSCAN_3_PCA, "DBSCAN_6_PCA", reduced_data, labels, True)
bench_clustering2(Agglomerative_1_PCA, "Agglomerative_1", reduced_data, labels, False)

init    	time	inertia	homo	compl	v-meas	ARI	AMI	silhouette	n_clusters
KMeans_1_PCA	0.037s	346	0.206	0.210	0.208	0.094	0.192	0.369	        8
init    	time	inertia	homo	compl	v-meas	ARI	AMI	silhouette	n_clusters
Kmeans_2_PCA	0.038s	346	0.203	0.207	0.205	0.092	0.189	0.384	        8
One or no clusters found
init    	time	homo	compl	v-meas	ARI	AMI	silhouette	n_clusters
DBSCAN_1_PCA	0.005s	nan	nan	nan	nan	nan	nan	         1
One or no clusters found
init    	time	homo	compl	v-meas	ARI	AMI	silhouette	n_clusters
DBSCAN_2_PCA	0.006s	nan	nan	nan	nan	nan	nan	         1
init    	time	homo	compl	v-meas	ARI	AMI	silhouette	n_clusters
DBSCAN_3_PCA	0.004s	0	0.283	0.284	0.068	0.212	0.088	        20
init    	time	homo	compl	v-meas	ARI	AMI	silhouette	n_clusters
DBSCAN_4_PCA	0.005s	0	0.116	0.011	0.000	0.005	0.386	         2
init    	time	homo	compl	v-meas	ARI	AMI	silhouette	n_clusters
DBSCAN_5_PCA	0.006s	0	0.335	0.010	0.000	0.005	0.435	         2
init    	time	homo	compl	v-meas	ARI	AMI	silhouette	n_clusters